# 1.0 Sobre o Notebook

- O objetivo deste notebook é fazer a limpeza e organização dos dados referentes ao dataset Olist
- Fonte dos dados: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
- As tabelas com dados brutos encontram-se na pasta data/raw/
- As tabelas limpas estão em data/processed/ 

# 2.0 Problema de Negócio

- A Olist apresenta crescimento nas vendas e satisfação de clientes que justifiquem investimentos na expansão da plataforma?
    - As vendas estão crescendo? -> Analisar a evolução mensal dos pedidos e valor vendido
    - Os clientes estão satisfeitos? -> Analisar review score
    - O que prejudica a satisfação de um cliente? -> Analisar prazo de entrega x review_score
    - Onde estão as oportunidades? -> Olhar categorias e regiões com melhor/pior desempenho

# 3.0 Imports

In [1]:
import pandas as pd

# 4.0 Funções

In [2]:
# caso não crie nenhuma função, vou apagar essa seção

# 5.0 Carregamento dos Dados Brutos

In [3]:
df_pedidos = pd.read_csv('../data/raw/olist_orders_dataset.csv')
df_clientes = pd.read_csv('../data/raw/olist_customers_dataset.csv')
df_itens = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
df_avaliacoes = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
df_produtos = pd.read_csv('../data/raw/olist_products_dataset.csv')

# 6.0 Limpeza das Tabelas

## 6.1 Tabela de Pedidos

In [4]:
# criando cópia dos dados raw para aplicar a limpeza
df_pedidos_clean = df_pedidos.copy()

In [5]:
# apagando colunas que não serão utilizadas
df_pedidos_clean = df_pedidos_clean.drop(columns=['order_approved_at', 'order_delivered_carrier_date'])

In [6]:
# checando limpeza
print(f'Dados Brutos: {df_pedidos.shape}')
print(f'Dados Limpos: {df_pedidos_clean.shape}')

Dados Brutos: (99441, 8)
Dados Limpos: (99441, 6)


In [7]:
# convertendo as colunas de data
df_pedidos_clean['order_purchase_timestamp'] = pd.to_datetime(df_pedidos_clean['order_purchase_timestamp'])
df_pedidos_clean['order_delivered_customer_date'] = pd.to_datetime(df_pedidos_clean['order_delivered_customer_date'])
df_pedidos_clean['order_estimated_delivery_date'] = pd.to_datetime(df_pedidos_clean['order_estimated_delivery_date'])

In [8]:
# checando ajuste das datas
print(f'Dados Brutos:\n {df_pedidos.dtypes}')
print()
print(f'Dados Limpos:\n {df_pedidos_clean.dtypes}')

Dados Brutos:
 order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

Dados Limpos:
 order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [9]:
df_pedidos_clean['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [10]:
# filtrando o dataset para manter apenas o status 'delivered'

df_pedidos_clean = df_pedidos_clean[df_pedidos_clean['order_status'] == 'delivered']

In [11]:
df_pedidos_clean['order_status'].value_counts()

order_status
delivered    96478
Name: count, dtype: int64

In [12]:
# checando limpeza
print(f'Dados Brutos: {df_pedidos.shape}')
print(f'Dados Limpos: {df_pedidos_clean.shape}')

Dados Brutos: (99441, 8)
Dados Limpos: (96478, 6)


In [13]:
# filtrando o período de análise de 2017-01 a 2018-08 (20 meses)
data_inicial = '2017-01-01 00:00:00'
data_final = '2018-08-31 23:59:59'

df_pedidos_clean = df_pedidos_clean[df_pedidos_clean['order_purchase_timestamp'].between(data_inicial, data_final)]

In [14]:
# analisando o período mês a mês após limpeza das pontas
temp_meses = df_pedidos_clean['order_purchase_timestamp']
print(f'Data inicial: {temp_meses.min()} e Data final: {temp_meses.max()}')

Data inicial: 2017-01-05 11:56:06 e Data final: 2018-08-29 15:00:37


In [15]:
# quantidade de pedidos mês a mês em ordem cronológica
temp_meses.dt.to_period('M').value_counts().sort_index()

order_purchase_timestamp
2017-01     750
2017-02    1653
2017-03    2546
2017-04    2303
2017-05    3546
2017-06    3135
2017-07    3872
2017-08    4193
2017-09    4150
2017-10    4478
2017-11    7289
2017-12    5513
2018-01    7069
2018-02    6555
2018-03    7003
2018-04    6798
2018-05    6749
2018-06    6099
2018-07    6159
2018-08    6351
Freq: M, Name: count, dtype: int64

In [16]:
# analisando os 8 nulos que haviam na coluna data de entrega para o cliente

data_nula = df_pedidos_clean['order_delivered_customer_date'].isna()
df_pedidos_clean.loc[data_nula, 'order_status'].value_counts()

order_status
delivered    8
Name: count, dtype: int64

In [17]:
# checando limpeza
print(f'Dados Brutos: {df_pedidos.shape}')
print(f'Dados Limpos: {df_pedidos_clean.shape}')

Dados Brutos: (99441, 8)
Dados Limpos: (96211, 6)


In [18]:
# salvando em data/processed
df_pedidos_clean.to_csv('../data/processed/pedidos_clean.csv', index=False)

### Registro da Limpeza - Tabela de Pedidos
- Colunas removidas: *order_approved_at*, *order_delivered_carrier_date* (8 -> 6 colunas)
- Datas convertidas: 3 colunas
- Filtro de status 'delivered': 99.441 -> 96.478 pedidos (-2.963)
- Filtro de período (jan/2017 a ago/2018): 96.478 -> 96.211 (-267)
    - Recorte confirmado após o filtro de status -> volume mensal consistente de jan/2017 a ago/2018 (6.351 pedidos em ago/2018)
- Mantidos os 8 pedidos sem data de entrega (serão excluídos apenas na análise de prazo)

## 6.2 Tabela de Clientes

In [19]:
# criando cópia dos dados raw para aplicar a limpeza
df_clientes_clean = df_clientes.copy()

In [20]:
# filtrando os dados de acordo com a tabela de pedidos
mascara_clientes_pedidos = df_clientes_clean['customer_id'].isin(df_pedidos_clean['customer_id'])
df_clientes_clean = df_clientes_clean[mascara_clientes_pedidos]

In [21]:
# checando filtro (deve ir para 96.211 linhas igual a tabela de pedidos)
print(f'Dados Brutos: {df_clientes.shape}')
print(f'Dados Limpos: {df_clientes_clean.shape}')

Dados Brutos: (99441, 5)
Dados Limpos: (96211, 5)


In [22]:
# apagando colunas que não serão utilizadas
df_clientes_clean = df_clientes_clean.drop(columns=['customer_zip_code_prefix', 'customer_city'])

In [23]:
# checando limpeza
print(f'Dados Brutos: {df_clientes.shape}')
print(f'Dados Limpos: {df_clientes_clean.shape}')

Dados Brutos: (99441, 5)
Dados Limpos: (96211, 3)


In [24]:
# contagem de customer_unique_id antes e depois das limpezas
print(f"Antes da limpeza: {df_clientes['customer_unique_id'].nunique()}")
print(f"Após a limpeza: {df_clientes_clean['customer_unique_id'].nunique()}")

Antes da limpeza: 96096
Após a limpeza: 93104


In [25]:
# salvando em data/processed
df_clientes_clean.to_csv('../data/processed/clientes_clean.csv', index=False)

### Registro da Limpeza - Tabela de Clientes
- Colunas removidas: *customer_zip_code_prefix*, *customer_city* (5 -> 3 colunas)
- Filtro dos dados de acordo com a Tabela de Pedidos: 99.441 -> 96.211 (-3.230)
    - Confirmada relação 1:1 entre as duas tabelas
- Clientes únicos: 96.096 -> 93.104 (-2.992)

## 6.3 Tabela de Itens dos Pedidos

In [37]:
# criando cópia dos dados raw para aplicar a limpeza
df_itens_clean = df_itens.copy()

In [38]:
# filtrando os dados de acordo com a tabela de pedidos
mascara_itens_pedidos = df_itens_clean['order_id'].isin(df_pedidos_clean['order_id'])
df_itens_clean = df_itens_clean[mascara_itens_pedidos]

In [39]:
# checando filtro
print(f'Dados Brutos: {df_itens.shape}')
print(f'Dados Limpos: {df_itens_clean.shape}')

Dados Brutos: (112650, 7)
Dados Limpos: (109880, 7)


In [40]:
# checando order_ids únicos (deve ir para 96.211)
df_itens_clean['order_id'].nunique()

96211

In [41]:
# apagando colunas que não serão utilizadas
df_itens_clean = df_itens_clean.drop(columns=['seller_id', 'shipping_limit_date', 'freight_value'])

In [42]:
# checando limpeza
print(f'Dados Brutos: {df_itens.shape}')
print(f'Dados Limpos: {df_itens_clean.shape}')

Dados Brutos: (112650, 7)
Dados Limpos: (109880, 4)


In [43]:
# salvando em data/processed
df_itens_clean.to_csv('../data/processed/itens_clean.csv', index=False)

### Registro da Limpeza - Tabela de Itens dos Pedidos
- Colunas removidas: *seller_id*, *shipping_limit_date* e *freight_value* (7 -> 4 colunas)
- Filtro de acordo com a Tabela de Pedidos: 112.650 -> 109.880 linhas de itens (-2.770)
- Previsão confirmada: 96.211 *order_id* únicos, todos os pedidos entregues possuem itens

## 6.4 Tabela de Avaliações

### Registro da Limpeza - Tabela de Avaliações
- Colunas removidas: 
- XXX

## 6.5 Tabela de Produtos

### Registro da Limpeza - Tabela de Produtos
- Colunas removidas: 
- XXX